<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Exercises_GOLD_VDB_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises GOLD: Text Summarization using NLP
Fill each TODO to build a graph-based extractive summarizer.

## What you'll learn
- Text cleaning (tokenize, lowercase, stopword removal).
- Word embeddings (GloVe) and sentence vectorization.
- Cosine similarity matrices and PageRank ranking.
- Extractive summarization over tennis articles.

## What you'll build
A graph-based summarizer that returns the top-ranked sentences for a set of articles.

## 0. Setup
Run installs once. If missing, download GloVe 100d from https://nlp.stanford.edu/data/glove.6B.zip and place glove.6B.100d.txt alongside the notebook.

In [4]:
%pip install --quiet pandas numpy nltk networkx
import pandas as pd
import numpy as np

In [2]:
import nltk
for res in ['punkt','punkt_tab','stopwords']:
    nltk.download(res, quiet=True)


## 🌟 Exercise 1 · Data loading and inspection

In [1]:
from pathlib import Path
import os
import urllib.request
import pandas as pd

data_path = 'tennis_articles.csv'
url = 'https://raw.githubusercontent.com/amit-sharma/Introduction-to-NLP-with-Python/master/data/tennis_articles_v4.csv'

if not os.path.exists(data_path):
    print(f"Tentative de téléchargement de {data_path}...")
    try:
        urllib.request.urlretrieve(url, data_path)
        print("Téléchargement réussi.")
    except Exception as e:
        print(f"Erreur de téléchargement : {e}")
        print("Création d'un jeu de données de secours pour continuer l'exercice...")
        data = {
            'article_title': ['Tennis 1', 'Tennis 2'],
            'article_text': [
                "Roger Federer is a Swiss professional tennis player. He has won 20 Grand Slam titles.",
                "Rafael Nadal is a Spanish professional tennis player. He is known as the King of Clay."
            ]
        }
        pd.DataFrame(data).to_csv(data_path, index=False, encoding='latin-1')

pdf = pd.read_csv(data_path, encoding='latin-1')
display(pdf.head())

if 'article_title' in pdf.columns:
    pdf = pdf.drop(columns=['article_title'])
print("Données prêtes.")

Tentative de téléchargement de tennis_articles.csv...
Erreur de téléchargement : HTTP Error 404: Not Found
Création d'un jeu de données de secours pour continuer l'exercice...


,article_title,article_text
0,Tennis 1,Roger Federer is a Swiss professional tennis p...
1,Tennis 2,Rafael Nadal is a Spanish professional tennis ...


Données prêtes.


## 🌟 Exercise 2 · Sentence tokenization

In [ ]:
import nltk
sentences_list = pdf['article_text'].apply(nltk.sent_tokenize).tolist()  # TODO: adjust column name if different
sentences = [s for doc in sentences_list for s in doc]
len(sentences), sentences[:3]


## 🌟 Exercise 3 · Load GloVe embeddings

In [3]:
%pip install --quiet kagglehub

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("danielwillgeorge/glove6b100dtxt")

print("Path to dataset files:", path)

100%|██████████| 131M/131M [00:01<00:00, 99.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/danielwillgeorge/glove6b100dtxt/versions/1


> Impotant note: after executing previous cell, open the path folder printed and verify that the file `glove.6B.100d.txt` is indeed there. Then copy it to the same folder as this notebook for the next steps.

In [6]:
import shutil
import kagglehub
from pathlib import Path
import os
import numpy as np

# S'assurer que le chemin est défini en téléchargeant si nécessaire
try:
    path_dir = Path(path)
except NameError:
    print("Variable 'path' non trouvée. Téléchargement via kagglehub...")
    path_dir = Path(kagglehub.dataset_download("danielwillgeorge/glove6b100dtxt"))

source_file = path_dir / 'glove.6B.100d.txt'
dest_file = Path('glove.6B.100d.txt')

if source_file.exists() and not dest_file.exists():
    shutil.copy(source_file, dest_file)
    print("Fichier GloVe copié localement.")

glove_path = dest_file if dest_file.exists() else source_file

if not glove_path.exists():
    raise FileNotFoundError('Le fichier glove.6B.100d.txt est introuvable.')

embeddings_index = {}
with glove_path.open('r', encoding='utf-8', errors='ignore') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

print(f"Vocabulaire GloVe chargé : {len(embeddings_index)} mots.")

Vocabulaire GloVe chargé : 400000 mots.


## 🌟 Exercise 4 · Text cleaning and normalization

In [10]:
import re
import nltk
from nltk.corpus import stopwords

# Téléchargement des ressources NLTK nécessaires
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# S'assurer que les phrases sont tokenisées
sentences_list = pdf['article_text'].apply(nltk.sent_tokenize).tolist()
sentences = [s for doc in sentences_list for s in doc]

stop_words = set(stopwords.words('english'))

def clean_sentence(s: str) -> str:
    s = s.lower()
    s = re.sub(r'[^a-z\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    tokens = [w for w in s.split() if w not in stop_words]
    return ' '.join(tokens)

cleaned_sentences = [clean_sentence(s) for s in sentences]
print(f"Nombre de phrases traitées : {len(cleaned_sentences)}")
cleaned_sentences[:3]

Nombre de phrases traitées : 4


['roger federer swiss professional tennis player',
 'grand slam titles',
 'rafael nadal spanish professional tennis player']

## 🌟 Exercise 5 · Sentence vectors

In [11]:
emb_dim = 100  # GloVe 100d
def sentence_vector(s: str):
    if not s:
        return np.zeros(emb_dim)
    words = s.split()
    vecs = [embeddings_index.get(w, np.zeros(emb_dim)) for w in words]
    return np.mean(vecs, axis=0)
sentence_vectors = np.array([sentence_vector(s) for s in cleaned_sentences])
sentence_vectors.shape


(4, 100)

## 🌟 Exercise 6 · Similarity matrix

In [12]:
from sklearn.metrics.pairwise import cosine_similarity
sim_mat = cosine_similarity(sentence_vectors)
sim_mat.shape


(4, 4)

## 🌟 Exercise 7 · Graph and PageRank

In [13]:
import networkx as nx
nx_graph = nx.from_numpy_array(sim_mat)
scores = nx.pagerank(nx_graph)
scores_list = sorted(((score, idx) for idx, score in scores.items()), reverse=True)
scores_list[:5]


[(0.2643085464787821, 0),
 (0.2625127074030964, 2),
 (0.2449519169610122, 1),
 (0.22822682915710907, 3)]

## 🌟 Exercise 8 · Summarization

In [15]:
top_n = 2  # Le résumé contiendra les 2 meilleures phrases
top_sentences = [sentences[idx] for _, idx in scores_list[:top_n]]

print(f"--- Résumé (Top {top_n} phrases) ---")
for s in top_sentences:
    print('-', s)

--- Résumé (Top 2 phrases) ---
- Roger Federer is a Swiss professional tennis player.
- Rafael Nadal is a Spanish professional tennis player.
